# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saibalaji2004/Flyrank-intern-repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis and time window

**Lane:** Content Refresh and Search Performance.

**Decision:** Identify content items that may need review because their Google Search performance declines.

**Unit of analysis:** One row in the source table represents one report date, one pseudonymized client, and one pseudonymized content item.

**Feature window:** February 1–March 31, 2026. Use February for historical comparison and March for the latest observed performance at the decision point.

**Target window:** April 1–30, 2026. Use this later period to define a future-decline proxy.

**Decision point:** March 31, 2026. Features must use only information available on or before this date. April data is reserved for creating the target, not the features.

**Source table:** `fact_content_daily_performance`, partitioned by month.

**Important distinction:** The source data is daily, but the analysis will aggregate it to one row per client and content item for the selected feature window. The target will be calculated separately from the following month.

In [6]:
%pip install -q duckdb

In [8]:

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is missing. Add it in Colab Secrets "
        "and enable notebook access."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

print("DuckDB is configured for the FlyRank warehouse.")


DuckDB is configured for the FlyRank warehouse.


In [16]:

feb = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_feb
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

mar = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_mar,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr_mar,
        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position_mar
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
""").df()

features = mar.merge(
    feb,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

features["click_change_feb_mar"] = (
    features["clicks_mar"] - features["clicks_feb"]
)

feature_cols = [
    "clicks_mar",
    "impressions_mar",
    "ctr_mar",
    "avg_position_mar",
    "click_change_feb_mar"
]

features = features.dropna(subset=feature_cols).copy()

print("Feature rows:", len(features))
display(features[feature_cols].head())


Feature rows: 86560


,clicks_mar,impressions_mar,ctr_mar,avg_position_mar,click_change_feb_mar
0,2.0,1140.0,0.001754,4.450877,-1.0
1,0.0,149.0,0.000000,5.637584,-1.0
2,6.0,1421.0,0.004222,6.906404,-1.0
3,16.0,2770.0,0.005776,3.950542,12.0
4,1.0,150.0,0.006667,7.506667,-1.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- `gsc_clicks`: observed Google Search clicks.
- `gsc_impressions`: observed Google Search impressions.
- `ctr_mar`: March click-through rate, calculated as clicks divided by impressions.
- `avg_position_mar`: impressions-weighted average search position in March.
- `click_change_feb_mar`: change in clicks from February to March.

These features use historical data available by March 31, 2026.

### Label
- `future_decline`: 1 if April clicks are at least 20% lower than March clicks; otherwise 0.

This is a proxy for declining search performance, not proof that content needs updating.

### Context
- `report_date`: date of the source record.
- `client_hash_id`: pseudonymized client identifier.
- `content_hash_id`: pseudonymized content identifier.

These fields identify and group records; they are not model features in this experiment.

### Excluded
- April clicks and impressions as features, because they belong to the future target window.
- Any target-derived columns from model inputs, because they cause data leakage.
- Client names, raw URLs, and private search queries, because they are unnecessary for this analysis.

In [17]:

apr = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_apr
    FROM {APR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

model_df = features.merge(
    apr,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["future_decline"] = (
    model_df["clicks_apr"] <= 0.8 * model_df["clicks_mar"]
).astype(int)

print("Eligible rows:", len(model_df))
print("Target distribution:")
display(model_df["future_decline"].value_counts())


Eligible rows: 86102
Target distribution:


,count
future_decline,
1,58815
0,27287


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:

# Recreate the Hugging Face authentication secret.
# HF_TOKEN should already be loaded from Colab Secrets.

con.execute("DROP SECRET IF EXISTS hf")
con.execute(
    "CREATE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face secret recreated.")


Hugging Face secret recreated.


In [11]:

display(
    con.execute(f"""
        SELECT COUNT(*) AS march_rows
        FROM read_parquet(
            '{FACT}/month=2026-03/*.parquet'
        )
    """).df()
)


,march_rows
0,9841378


In [12]:

MARCH = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

display(
    con.execute(
        f"DESCRIBE SELECT * FROM {MARCH}"
    ).df()
)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [13]:

display(
    con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT report_date) AS distinct_dates,
            COUNT(DISTINCT client_hash_id) AS clients,
            COUNT(DISTINCT content_hash_id) AS content_items,
            COUNT(DISTINCT (
                report_date,
                client_hash_id,
                content_hash_id
            )) AS distinct_grain_keys
        FROM {MARCH}
    """).df()
)


,total_rows,distinct_dates,clients,content_items,distinct_grain_keys
0,9841378,31,55,331437,9841378


In [14]:

display(
    con.execute(f"""
        SELECT
            MIN(report_date) AS first_date,
            MAX(report_date) AS last_date,
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (
                WHERE gsc_clicks IS NOT NULL
            ) AS rows_with_clicks,
            COUNT(*) FILTER (
                WHERE gsc_impressions IS NOT NULL
            ) AS rows_with_impressions,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS gsc_available_rows
        FROM {MARCH}
    """).df()
)


,first_date,last_date,total_rows,rows_with_clicks,rows_with_impressions,gsc_available_rows
0,2026-03-01,2026-03-31,9841378,9841378,9841378,3611061


In [15]:

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
APR = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

display(
    con.execute(f"""
        SELECT 'February features' AS window,
               MIN(report_date) AS first_date,
               MAX(report_date) AS last_date,
               COUNT(*) AS total_rows
        FROM {FEB}

        UNION ALL

        SELECT 'March features' AS window,
               MIN(report_date) AS first_date,
               MAX(report_date) AS last_date,
               COUNT(*) AS total_rows
        FROM {MARCH}

        UNION ALL

        SELECT 'April target' AS window,
               MIN(report_date) AS first_date,
               MAX(report_date) AS last_date,
               COUNT(*) AS total_rows
        FROM {APR}
    """).df()
)


,window,first_date,last_date,total_rows
0,February features,2026-02-01,2026-02-28,7355108
1,March features,2026-03-01,2026-03-31,9841378
2,April target,2026-04-01,2026-04-30,10424730


In [18]:

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = model_df[feature_cols]
y = model_df["future_decline"]

if y.nunique() < 2:
    print("Leakage experiment skipped: only one target class.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    honest_model = DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    )
    honest_model.fit(X_train, y_train)
    honest_score = accuracy_score(
        y_test, honest_model.predict(X_test)
    )

    X_leaky = X.copy()
    X_leaky["leaked_label"] = y

    XL_train, XL_test, yL_train, yL_test = train_test_split(
        X_leaky, y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    leaky_model = DecisionTreeClassifier(random_state=42)
    leaky_model.fit(XL_train, yL_train)
    leaky_score = accuracy_score(
        yL_test, leaky_model.predict(XL_test)
    )

    print(f"Honest accuracy: {honest_score:.3f}")
    print(f"Leaky accuracy: {leaky_score:.3f}")
    print("The leaked-label column is intentionally excluded from real features.")


Honest accuracy: 0.703
Leaky accuracy: 1.000
The leaked-label column is intentionally excluded from real features.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- The warehouse has an unbalanced history, so not every client or content item has the same amount of data.
- Google Search Console and Google Analytics 4 availability can differ. Missing metrics must not automatically be treated as zero.
- April measurements belong to the future target window and must not be used as features.
- A decline in clicks does not identify its cause. Seasonality, search demand, competition, and ranking changes may all contribute.
- The 20% decline threshold is a chosen proxy, not a validated definition of content quality.
- The results are observational and support directional prioritization. They do not prove causality or guarantee future performance.

In [19]:

display(
    con.execute(f"""
        SELECT
            MIN(report_date) AS first_date,
            MAX(report_date) AS last_date,
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS gsc_available_rows,
            COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            ) AS ga4_available_rows
        FROM {MARCH}
    """).df()
)


,first_date,last_date,total_rows,gsc_available_rows,ga4_available_rows
0,2026-03-01,2026-03-31,9841378,3611061,413966


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.